<a href="https://colab.research.google.com/github/NahomKidane/ai301-labs/blob/main/m05-clustering-and-topic-modeling/guided-labs/AI301_Guided_Lab_Part0_Concept_Dimensionality_Reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI 301 Guided Lab: Clustering and Topic Modeling, Part 0 · Dimensionality Reduction

**Module 5 · Clustering and Topic Modeling**

> **File > Save a copy in Drive** before you start. Leave the runtime on CPU: every method here
> runs in seconds and none of them uses a GPU. Nothing is downloaded; the dataset ships with
> scikit-learn.

Part 1 of this lab reduces 384-dimensional text embeddings to 5 numbers with UMAP before clustering
them. This notebook shows why that step exists and what the three common methods do with the same
data:

- **PCA** keeps the directions along which the data spreads most, Part 3.
- **t-SNE** keeps each point next to its nearest neighbours, Part 4.
- **UMAP** keeps the neighbors and the graph that links them, Part 5.

The data is 1,797 images of handwritten digits. Each image is 8 by 8 pixels, so each one is a
vector of 64 numbers, the way each abstract in Part 1 is a vector of 384 numbers. The digit each
image shows is known, so you can see which method keeps the ten groups apart.

In [ ]:
%pip install -q umap-learn hdbscan

---

## Part 0 · Setup

In [ ]:
# UMAP and scikit-learn print warnings about parallelism and renamed arguments, neither is an error
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_map(points, labels, title):
    """Scatter the 2D points, one colour per digit, with a legend."""
    plt.figure(figsize=(7, 6))
    for digit in range(10):
        keep = labels == digit
        plt.scatter(points[keep, 0], points[keep, 1], s=8, label=str(digit))
    plt.legend(title="digit", markerscale=2, loc="best")
    plt.title(title)
    plt.xticks([])
    plt.yticks([])
    plt.show()

---

## Part 1 · The data

`load_digits` returns the images already flattened: one row per image, 64 columns, one column per
pixel, each value from 0 (white) to 16 (black). `target` holds the digit each image shows.

- **Python note:** `X[i].reshape(8, 8)` turns the 64 numbers of image `i` back into an 8 by 8 grid
  so `imshow` can draw it. `np.bincount` counts how many images there are of each digit.

> **Predict before running.**
>
> Two images of the digit 1, written by different people. Roughly how many of their 64 pixel values
> will match?

**Your prediction:**

_Type here._

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data
y = digits.target

print("shape:", X.shape)
print("pixel values from", X.min(), "to", X.max())
print("images per digit:", np.bincount(y))
print()
print("image 0 as 64 numbers:")
print(X[0])

# the first ten images, one per row of the figure
plt.figure(figsize=(10, 1.4))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X[i].reshape(8, 8), cmap="gray_r")
    plt.title(str(y[i]))
    plt.axis("off")
plt.show()

**Interpret the output.**

1. Say what one row of `X` is and what one column of `X` is. Then say which of the two an embedding
   model's output dimension corresponds to.
2. The ten pictures are readable to you at 64 pixels. Say what a method that keeps only two numbers
   per image would have to throw away, and whether the digit identity is likely to survive.

**Your answer:**

_Type here._

---

## Part 2 · Distances in many dimensions

Clustering works from distances between points. The curse of dimensionality figure in Part 1 shows
what happens to those distances as dimensions grow: for random points, the nearest pair and the
farthest pair end up almost the same distance apart, so "nearest" stops meaning anything.

The cell below reproduces that figure's bars with code. It draws 1,000 random points in 2, 10, 64
and 784 dimensions and divides the nearest pair's distance by the farthest pair's. It then does the
same for 1,000 of the digit images in their 64 dimensions.

- **Python note:** `pairwise_distances` returns a 1,000 by 1,000 grid of every distance. The
  diagonal is a point's distance to itself, 0, so it is set to `np.inf` before taking the minimum.
  `D[D < np.inf]` keeps every entry except those.

> **Predict before running.**
>
> The digits are also 64 numbers each. Will their ratio match the random 64 dimensional points,
> or be closer to the 2 dimensional ones?

**Your prediction:**

_Type here._

In [ ]:
from sklearn.metrics import pairwise_distances


def nearest_over_farthest(points):
    """Distance of the closest pair divided by the distance of the farthest pair."""
    D = pairwise_distances(points)
    D[np.diag_indices_from(D)] = np.inf
    return D.min() / D[D < np.inf].max()


rng = np.random.RandomState(42)

rows = []
for dims in [2, 10, 64, 784]:
    random_points = rng.uniform(size=(1000, dims))
    rows.append({"data": "random", "dimensions": dims, "nearest over farthest": round(nearest_over_farthest(random_points), 3)})

# 1,000 digit images chosen at random
sample = rng.choice(len(X), 1000, replace=False)
rows.append({"data": "digits", "dimensions": 64, "nearest over farthest": round(nearest_over_farthest(X[sample]), 3)})

pd.DataFrame(rows)

**Interpret the output.**

1. The random points in 64 dimensions give about 0.5, the digits give about 0.12. Say what that
   difference tells you about how the digit images fill their 64 dimensions, and why that is the
   thing a reduction method can exploit.
2. Text embeddings from gte-small are 384 numbers each. Say whether you expect their ratio to look
   like the random 784 row or like the digits row, and what would have to be true of the embedding
   model for each.

**Your answer:**

_Type here._

---

## Part 3 · PCA

PCA finds the direction along which the points spread out most, then the direction at right angles
to it with the next most spread, and so on. Keeping the first two directions gives every image two
numbers: how far along each direction it sits. Nothing else is stored.

Each direction is a fixed combination of the 64 pixels, so a new image can be placed on the same
map without refitting, and the same input always gives the same output.

`explained_variance_ratio_` reports how much of the total spread each kept direction holds. The
rest is discarded.

- **Python note:** `fit_transform` learns the directions from `X` and returns the two numbers per
  image in one call. `n_components` is how many directions to keep.

> **Predict before running.**
>
> 1. Out of 64 directions, how much of the spread will the first two hold: most of it, about half,
>    or well under half?
> 2. Will the ten digits form ten separate groups in the map?

**Your prediction:**

_Type here._

In [ ]:
from sklearn.decomposition import PCA

start = time.time()
pca = PCA(n_components=2, random_state=42)
pca_points = pca.fit_transform(X)
print("seconds:", round(time.time() - start, 2))

print("spread held by each direction:", np.round(pca.explained_variance_ratio_, 3))
print("spread held by both:", round(pca.explained_variance_ratio_.sum(), 3))

plot_map(pca_points, y, "PCA, two directions")

Two directions keep about 28 percent of the spread. The cell below fits PCA with all 64 directions
and counts how many are needed to reach half and 90 percent.

- **Python note:** `np.cumsum` adds the values up as it goes, so position `i` holds the spread kept
  by the first `i + 1` directions. `np.argmax(... >= 0.9)` returns the first position where that
  running total reaches 0.9.

In [ ]:
pca_all = PCA().fit(X)
running_total = np.cumsum(pca_all.explained_variance_ratio_)

print("directions needed for half the spread:", int(np.argmax(running_total >= 0.5)) + 1)
print("directions needed for 90 percent:", int(np.argmax(running_total >= 0.9)) + 1)

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, 65), running_total, marker=".")
plt.xlabel("directions kept")
plt.ylabel("share of spread kept")
plt.grid(alpha=0.3)
plt.show()

**Interpret the output.**

1. Name two digits that overlap in the PCA map and two that separate. Say what about their shapes
   would make the first pair look alike along the two directions of largest spread.
2. Five directions hold half the spread and 21 hold 90 percent. Say what you would lose by giving
   HDBSCAN the 5 number version instead of the 21 number version, and what you would gain.
3. PCA is the only method in this notebook whose two axes have a meaning. Say what the meaning is,
   and name a situation where that matters more than how well the groups separate.

**Your answer:**

_Type here._

---

## Part 4 · t-SNE

t-SNE ignores the overall spread and looks only at neighbourhoods. For each image it works out
which other images are its close neighbours in 64 dimensions, then places points on the map so that
those same images are close in two dimensions. Points that were far apart are pushed apart, but how
far apart is not controlled.

Two consequences follow:

- Distances between groups on the map mean little. Two groups drawn far apart may or may not be
  far apart in the data.
- The map starts from a random layout, so a different `random_state` gives a different picture.

`perplexity` sets roughly how many neighbours each point tries to keep. The default is 30.

- **Python note:** `TSNE` has no `transform` method. It can only map the points it was fitted on,
  so a new image cannot be placed on an existing map.

> **Predict before running.**
>
> 1. Will the ten digits separate better or worse than in the PCA map?
> 2. Rerun with a different `random_state`. Will the groups be in the same places?

**Your prediction:**

_Type here._

In [ ]:
from sklearn.manifold import TSNE

start = time.time()
tsne_points = TSNE(n_components=2, perplexity=30, random_state=42, init="pca").fit_transform(X)
print("seconds:", round(time.time() - start, 1))

plot_map(tsne_points, y, "t-SNE, perplexity 30, random_state 42")

In [ ]:
# the same data, a different starting layout
other_seed = TSNE(n_components=2, perplexity=30, random_state=0, init="pca").fit_transform(X)
plot_map(other_seed, y, "t-SNE, perplexity 30, random_state 0")

# the same data, fewer neighbours per point
few_neighbours = TSNE(n_components=2, perplexity=5, random_state=42, init="pca").fit_transform(X)
plot_map(few_neighbours, y, "t-SNE, perplexity 5, random_state 42")

**Interpret the output.**

1. Compare the two seeds. Say what stayed the same and what moved, and what that means for a
   sentence such as "the 3s are close to the 5s" written from a t-SNE map.
2. At perplexity 5 some digits split into more than one island. Say why keeping fewer neighbours
   would do that, and whether a clustering method run on that map would report the right number of
   groups.
3. A colleague sends you a t-SNE map of customer reviews with no seed and no perplexity stated. Say
   what you can and cannot conclude from it.

**Your answer:**

_Type here._

---

## Part 5 · UMAP

UMAP also starts from each point's neighbours, but it builds a graph from them first, one edge per
neighbour pair, and then lays that graph out in two dimensions. Because the whole graph is laid out
together, the positions of groups relative to each other mean more than in t-SNE, though still less
than in PCA.

Two settings matter:

| Argument | Default | What it does |
|---|---|---|
| `n_neighbors` | `15` | how many neighbours each point is linked to; small values keep local detail, large values keep the overall shape |
| `min_dist` | `0.1` | how close points may sit on the map; `0.0` lets a group collapse into a tight blob, which suits clustering |

UMAP has a `transform` method, so new points can be placed on a fitted map, and `random_state`
makes a rerun reproducible. Part 1 of this lab uses `n_components=5` and `min_dist=0.0` on the
abstracts for exactly this reason.

- **Python note:** the first UMAP call in a session compiles its code and takes longer. Later calls
  in the same session run in a few seconds.

> **Predict before running.**
>
> 1. Will the UMAP map look more like the PCA map or the t-SNE map?
> 2. What will `min_dist=0.0` do to the shape of each group?

**Your prediction:**

_Type here._

In [ ]:
from umap import UMAP

start = time.time()
umap_points = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X)
print("seconds:", round(time.time() - start, 1))

plot_map(umap_points, y, "UMAP, n_neighbors 15, min_dist 0.1")

In [ ]:
# fewer neighbours: more local detail, less overall shape
few = UMAP(n_components=2, n_neighbors=5, min_dist=0.1, random_state=42).fit_transform(X)
plot_map(few, y, "UMAP, n_neighbors 5, min_dist 0.1")

# min_dist 0: groups collapse into tight blobs, the setting Part 1 uses before clustering
tight = UMAP(n_components=2, n_neighbors=15, min_dist=0.0, random_state=42).fit_transform(X)
plot_map(tight, y, "UMAP, n_neighbors 15, min_dist 0.0")

**Interpret the output.**

1. Compare the three UMAP maps with the t-SNE maps. Say which method's groups moved less between
   settings, and why the way UMAP lays out its graph would explain that.
2. With `min_dist=0.0` the groups are tight and the gaps between them are wide. Say why that is
   what a density based clustering method wants, and what a person reading the map loses.
3. Part 1 reduces to 5 dimensions, not 2, before clustering. Say what a 2D map is for and what a
   5D reduction is for, and why the two jobs want different settings.

**Your answer:**

_Type here._

---

## Part 6 · Side by side

The three default maps in one figure, then a table of what each method keeps and what it costs.

In [ ]:
maps = [
    ("PCA", pca_points),
    ("t-SNE", tsne_points),
    ("UMAP", umap_points),
]

plt.figure(figsize=(15, 4.6))
for i in range(3):
    name, points = maps[i]
    plt.subplot(1, 3, i + 1)
    for digit in range(10):
        keep = y == digit
        plt.scatter(points[keep, 0], points[keep, 1], s=5, label=str(digit))
    plt.title(name)
    plt.xticks([])
    plt.yticks([])
plt.legend(title="digit", markerscale=3, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

| | PCA | t-SNE | UMAP |
|---|---|---|---|
| Keeps | directions of largest spread | each point's neighbours | neighbours and the graph between them |
| Axes mean something | yes | no | no |
| Distance between groups means something | yes | no | roughly |
| Same result on rerun | yes | only with a seed | only with a seed |
| Can place a new point on the map | yes | no | yes |
| Seconds on 1,797 images | under 1 | about 5 | about 4 after the first call |
| Feeds HDBSCAN in Part 1 | | | yes, at 5 dimensions |

**Interpret the output.**

1. Say which map you would put in a report for someone who has never seen the data, and which you
   would give a clustering method. If they differ, say what each reader needs that the other does
   not.
2. A method that cannot place new points has a cost in production. Say what that cost is for a
   system that clusters incoming support tickets every day.

**Your answer:**

_Type here._

> **Your turn.**
>
> Run the Part 1 pipeline on the digits: UMAP to 5 dimensions with `min_dist=0.0`, then HDBSCAN,
> then score the cluster labels against the true digits with `adjusted_rand_score` and
> `normalized_mutual_info_score`, leaving the outliers out. Try `min_cluster_size` of 10, 20 and 50
> and record the cluster count, the outlier count and both scores for each. Say which value you
> would choose and why, given that there are ten digits.
>
> Both scores are 1 for a perfect match. ARI drops below 0 for a grouping worse than chance, while
> NMI stops at 0. NMI is also kinder to a clustering that splits one digit into several clusters,
> so when the two disagree, look at the cluster count. Purity and F-measure are two more labelled
> scores you will meet in papers; scikit-learn has no direct call for them.

In [ ]:
from hdbscan import HDBSCAN
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Your turn: reduce to 5 dimensions, cluster, and score against the true digits.

---

## Check yourself

- Say what PCA keeps and what t-SNE keeps, in one sentence each.
- Say why two t-SNE maps of the same data can look different.
- Say which method can place a new point on an existing map without refitting.
- Say why Part 1 reduces to 5 dimensions for clustering and to 2 for the picture.

## Summary

Each image is a vector of 64 numbers, the way each abstract is a vector of 384. In many dimensions
the nearest and farthest points end up at similar distances, and real data escapes that only because
it does not fill its dimensions evenly.

PCA keeps the directions of largest spread. Its axes mean something and it places new points, but
two directions hold under a third of the spread here and the digits overlap.

t-SNE and UMAP keep neighbourhoods, so the groups separate. t-SNE's layout changes with the seed and
its group distances mean little. UMAP lays out a graph, holds its shape better across settings, and
can place new points.

For clustering, UMAP to a few dimensions with `min_dist=0.0` gives tight groups with gaps between
them, which is what a density based method looks for.

## References

- [scikit-learn, load_digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html),
  the UCI Optical Recognition of Handwritten Digits dataset
- [scikit-learn, PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)
- [scikit-learn, TSNE](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html)
- [UMAP documentation](https://umap-learn.readthedocs.io/en/latest/), including
  [Basic UMAP Parameters](https://umap-learn.readthedocs.io/en/latest/parameters.html)
- [Understanding UMAP](https://pair-code.github.io/understanding-umap/), Google PAIR
- Wattenberg, Viégas and Johnson, [How to Use t-SNE Effectively](https://distill.pub/2016/misread-tsne/), Distill, 2016
- [How HDBSCAN Works](https://hdbscan.readthedocs.io/en/latest/how_hdbscan_works.html)
- [sklearn clustering performance evaluation](https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation), the adjusted Rand index, NMI and the unlabelled scores